## 1. General Housekeeping
- Importing libraries and setting up the charts. Everything is preinstalled in Colab.
- The regressions are done with numpy so every step is visible.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

## Housekeeping for the charts
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (12, 5.5),
    "figure.dpi": 110,
    "axes.titlesize": 15,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
    "font.size": 10.5,
})

## 2. Question and General Parameters
**When Ireland changed the VAT rate on hospitality, how much of the change reached customers' prices?**

The hospitality VAT rate has moved between 13.5% and 9% several times:

| Date | Change | Why |
|---|---|---|
| 1 July 2011 | Cut 13.5% → 9% | Jobs Initiative after the financial crisis |
| 1 January 2019 | Raised 9% → 13.5% | Reversal of the temporary cut |
| 1 November 2020 | Cut 13.5% → 9% | COVID-19 support |
| 1 September 2023 | Raised 9% → 13.5% | Reversal of the temporary cut |
| 1 July 2026 | Cut 13.5% → 9% (food-led hospitality and hairdressing only, not hotels) | Budget 2026 |

**Pass-through** is the share of a tax change that shows up in prices. Moving between 9% and 13.5% VAT changes a VAT-inclusive price by about 4%, so:
- **100% pass-through** → prices move the full ~4%
- **0% pass-through** → prices don't move and businesses absorb (or keep) the change

**Method: event study (difference-in-differences).** Around each change, I compare prices of the **affected** services (restaurant food, hotels, hairdressing) with **unaffected** services from the same CPI (e.g. alcohol in pubs, which stays at 23% VAT, car repairs, dental services). The unaffected services show what would have happened to prices anyway.

**Benchmark:** the Irish Fiscal Advisory Council (Carroll, 2025, Working Paper No. 27) studied the same four changes using detailed CSO and UK price data and found pass-through of 50% (2011 cut), 88% (2019 rise), 6% (2020 cut) and 36% (2023 rise): increases were passed on more than cuts.

In [ ]:
FULL_PASS_THROUGH = np.log(1.135 / 1.09)   ## ≈ 0.0405: log price change if a 9% ↔ 13.5% VAT change is fully passed on
WINDOW = 6                                 ## months before and after each change

## First month at the new rate. 'exclude' removes treated categories not covered by that change
EVENTS = [
    {"name": "2011 cut",  "month": "2011-07", "direction": -1, "exclude": []},
    {"name": "2019 rise", "month": "2019-01", "direction": +1, "exclude": []},
    {"name": "2020 cut",  "month": "2020-11", "direction": -1, "exclude": []},
    {"name": "2023 rise", "month": "2023-09", "direction": +1, "exclude": []},
    {"name": "2026 cut",  "month": "2026-07", "direction": -1, "exclude": ["accommodation", "hotel"]},   ## hotels stayed at 13.5%
]

## Fiscal Council benchmark pass-through (Carroll 2025, Table 2)
BENCHMARK = {"2011 cut": 0.50, "2019 rise": 0.88, "2020 cut": 0.06, "2023 rise": 0.36}

## Which CPI sub-indices count as affected (treated) and unaffected (control). Matched by keywords, case-insensitive
TREATED_KEYWORDS = ["restaurant", "cafe", "café", "canteen", "accommodation", "hotel", "hairdress"]
CONTROL_KEYWORDS = ["alcoholic drink", "repair", "dental", "veterinar", "cleaning", "domestic services",
                    "medical services", "paramedical"]
EXCLUDE_KEYWORDS = ["restaurants & hotels", "restaurants and hotels", "restaurants & accommodation",
                    "restaurants and accommodation", "all items"]     ## Broad aggregates that mix treated and untreated items
MIN_POST_MONTHS = 2                        ## Skip an event if fewer months of data after it

## Allocated colours, used throughout
TREATED_COLOR, CONTROL_COLOR = "#2563eb", "#9ca3af"
CUT_COLOR, RISE_COLOR = "#16a34a", "#dc2626"

## 3. Load the Data
The CSO blocks downloads from Colab's servers, so I download the table in my browser and upload it here.

**Which table:** on data.cso.ie, search for **"Consumer Price Index by Detailed Sub Indices"**. Select the **index** statistic (not the percentage changes), **all months**, and **all sub-indices**, then download as CSV. If the file is too big, select only the sub-indices printed below as treated and control.

In [ ]:
path = "cpi.csv"
if not os.path.exists(path):
    from google.colab import files
    uploaded = files.upload()                   ## Opens a "Choose files" button: pick the CSV you downloaded
    os.rename(next(iter(uploaded)), path)       ## Rename it so the rest of the notebook finds it
raw = pd.read_csv(path)
print("Columns:", list(raw.columns))

def find_col(keyword):
    '''Find the column whose name contains a keyword (the CSO's column names include codes that can change).'''
    matches = [c for c in raw.columns if keyword.lower() in c.lower()]
    if not matches:
        raise KeyError(f"No column containing '{keyword}'. Columns are: {list(raw.columns)}")
    return matches[-1]     ## Label columns come after code columns in CSO files

MONTH, STAT = find_col("Month"), find_col("Statistic Label")
## The category column is the label column with the most distinct values (e.g. "Detailed Sub Indices")
label_cols = [c for c in raw.columns if c not in (MONTH, STAT, "UNIT", "VALUE") and not pd.api.types.is_numeric_dtype(raw[c])
              and not re.match(r"^(C\d|TLIST|STATISTIC)", c)]
CATEGORY = max(label_cols, key=lambda c: raw[c].nunique())

def parse_month(text):
    '''Handle CSO month formats like "2019M01", "2019 January" or "2019 Jan".'''
    text = str(text)
    m = re.match(r"(\d{4})\s*M(\d{1,2})$", text)
    if m:
        return pd.Period(f"{m[1]}-{int(m[2]):02d}", freq="M")
    return pd.Period(pd.to_datetime(text, format="mixed"), freq="M")

cpi = raw[raw[STAT].str.contains("index", case=False) & ~raw[STAT].str.contains("change|%", case=False)].copy()
cpi["month"] = cpi[MONTH].map(parse_month)
cpi["value"] = pd.to_numeric(cpi["VALUE"], errors="coerce")
cpi = cpi.dropna(subset=["value"]).rename(columns={CATEGORY: "series", STAT: "stat"})[["series", "stat", "month", "value"]]
print(f"Category column: '{CATEGORY}'. Index statistics found: {cpi['stat'].unique().tolist()}")
print(f"{cpi['series'].nunique()} sub-indices, {cpi['month'].min()} → {cpi['month'].max()}")

### Pick the Treated and Control Sub-Indices
Matched by keyword. **Check this printout**: if a series is in the wrong group, edit the keyword lists in section 2 and rerun.

Note on restaurants: if the CSO splits **food** and **alcoholic drinks** consumed in restaurants and pubs, the food series is treated and the alcohol series is a control. That matters, because alcohol stayed at the 23% standard rate throughout.

In [ ]:
def has(label, keywords):
    return any(k in label.lower() for k in keywords)

all_series = sorted(cpi["series"].unique())
EXCLUDED = [s for s in all_series if has(s, EXCLUDE_KEYWORDS)]
CONTROLS = [s for s in all_series if has(s, CONTROL_KEYWORDS) and s not in EXCLUDED]
TREATED = [s for s in all_series if has(s, TREATED_KEYWORDS) and s not in EXCLUDED + CONTROLS]

print("TREATED (hospitality VAT rate):"); [print("   ", s) for s in TREATED]
print("\nCONTROL (VAT rate unchanged):"); [print("   ", s) for s in CONTROLS]
print("\nLeft out (broad aggregates):"); [print("   ", s) for s in EXCLUDED]
if not TREATED or len(CONTROLS) < 2:
    print("\n⚠️ Too few series matched. All sub-indices in the file:", all_series)

### Chart 1: Affected vs. Unaffected Prices Over Time
Average of the treated and control sub-indices (log scale, each rebased to its first month so only **growth** is compared). Shaded periods are when hospitality VAT was 9%. If VAT changes are passed on, the blue line should bend down at the start of each shaded period and up at the end, relative to grey.

In [ ]:
## Use the index statistic with the longest coverage for this overview
main_stat = cpi.groupby("stat")["month"].nunique().idxmax()
wide = cpi[cpi["stat"] == main_stat].pivot_table(index="month", columns="series", values="value")
logs = np.log(wide)
rebased = logs - logs.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else np.nan)
groups = pd.DataFrame({"Affected (treated)": rebased[[s for s in TREATED if s in rebased]].mean(axis=1),
                       "Unaffected (control)": rebased[[s for s in CONTROLS if s in rebased]].mean(axis=1)}).dropna()

LOW_VAT = [("2011-07", "2018-12"), ("2020-11", "2023-08"), ("2026-07", str(groups.index.max()))]
fig, ax = plt.subplots()
x = groups.index.to_timestamp()
ax.plot(x, groups["Affected (treated)"] * 100, color=TREATED_COLOR, lw=2.2, label="Affected by hospitality VAT")
ax.plot(x, groups["Unaffected (control)"] * 100, color=CONTROL_COLOR, lw=2.2, label="Unaffected services")
for start, end in LOW_VAT:
    s, e = pd.Period(start, "M").to_timestamp(), pd.Period(end, "M").to_timestamp()
    if s <= x.max() and e >= x.min():
        ax.axvspan(max(s, x.min()), min(e, x.max()), color=CUT_COLOR, alpha=0.07)
ax.text(0.01, 0.97, "Shaded = 9% VAT", transform=ax.transAxes, color=CUT_COLOR, fontsize=9.5, va="top")
ax.set_title(f"Price growth since {groups.index.min()}: affected vs. unaffected services")
ax.set_ylabel("Log points since start (≈ % change)")
ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

## 4. Event Studies
For each VAT change, using the 6 months before and after:

log(price) = series effect + month effect + Σ β_k × (affected × k months from the change)

- **Series effects** absorb permanent differences between services
- **Month effects** absorb anything hitting all prices at once (general inflation)
- **β_k** is how far affected prices moved relative to unaffected ones, k months from the change, compared with the month just before (k = -1)

**Pass-through** = the average β over the 6 months from the change ÷ the full 4% effect. Before the change, β should be near zero if affected and unaffected prices were moving in parallel.

**Checking for luck:** I rerun each event pretending each control series was the affected one (a **placebo test**). The grey band shows the range of those fake effects.

In [ ]:
def fit_ols(X, y):
    '''Ordinary least squares with numpy.'''
    beta, *_ = np.linalg.lstsq(X.values, y.values, rcond=None)
    return pd.Series(beta, index=X.columns)

def event_data(event, treated):
    '''Log prices for treated + control series in the window, using the index statistic with the best coverage.'''
    start = pd.Period(event["month"], "M") - WINDOW
    end = pd.Period(event["month"], "M") + (WINDOW - 1)
    series = treated + CONTROLS
    d = cpi[cpi["series"].isin(series) & (cpi["month"] >= start) & (cpi["month"] <= end)]
    if d.empty:
        return None
    stat = d.groupby("stat").size().idxmax()                 ## CSO may publish several base periods
    d = d[d["stat"] == stat].copy()
    d["k"] = [(m - pd.Period(event["month"], "M")).n for m in d["month"]]
    d["log_price"] = np.log(d["value"])
    return d

def event_study(d, treated):
    '''β_k for the treated group relative to k = -1. Returns the β series.'''
    X = pd.concat([pd.Series(1.0, index=d.index, name="const"),
                   pd.get_dummies(d["series"], prefix="s", drop_first=True, dtype=float),
                   pd.get_dummies(d["month"].astype(str), prefix="m", drop_first=True, dtype=float)], axis=1)
    is_treated = d["series"].isin(treated).astype(float)
    ks = sorted(d["k"].unique())
    for k in ks:
        if k != -1:
            X[f"k_{k}"] = is_treated * (d["k"] == k)
    b = fit_ols(X, d["log_price"])
    return pd.Series({k: (0.0 if k == -1 else b[f"k_{k}"]) for k in ks})

def pass_through(betas, event):
    post = betas[betas.index >= 0]
    return post.mean() / (event["direction"] * FULL_PASS_THROUGH)

results = {}
for ev in EVENTS:
    treated = [s for s in TREATED if not has(s, ev["exclude"])]
    d = event_data(ev, treated)
    if d is None or d["k"].min() > -2 or d["k"].max() < MIN_POST_MONTHS - 1 or not d["series"].isin(treated).any():
        print(f"⏭️ {ev['name']}: not enough data in the file, skipped")
        continue
    betas = event_study(d, treated)
    ## Placebos: drop the real treated series, pretend each control was affected
    d_ctrl = d[~d["series"].isin(treated)]
    placebo = {c: event_study(d_ctrl, [c]) for c in CONTROLS if c in d_ctrl["series"].unique()}
    placebo_pt = np.array([pass_through(p, ev) for p in placebo.values()])
    pt = pass_through(betas, ev)
    ## Each affected category on its own
    by_series = {s: pass_through(event_study(d[d["series"].isin([s] + CONTROLS)], [s]), ev)
                 for s in treated if s in d["series"].unique()}
    results[ev["name"]] = {"event": ev, "betas": betas, "placebo": pd.DataFrame(placebo), "pass_through": pt,
                           "placebo_p": (np.abs(placebo_pt) >= abs(pt)).mean(), "by_series": by_series,
                           "post_months": int(d["k"].max()) + 1}
    print(f"✅ {ev['name']}: pass-through {pt:.0%} ({results[ev['name']]['post_months']} months after the change)")

### Chart 2: Prices Around Each VAT Change
Blue: affected prices relative to unaffected ones, month by month (month before the change = 0). Grey band: range of the placebo series. The dashed line marks where prices would be with **full** pass-through.

In [ ]:
n = len(results)
fig, axes = plt.subplots(1, n, figsize=(4.2 * n, 4.8), sharey=True)
axes = np.atleast_1d(axes)
for ax, (name, r) in zip(axes, results.items()):
    ev, b = r["event"], r["betas"]
    color = CUT_COLOR if ev["direction"] < 0 else RISE_COLOR
    if not r["placebo"].empty:
        ax.fill_between(b.index, r["placebo"].min(axis=1) * 100, r["placebo"].max(axis=1) * 100,
                        color=CONTROL_COLOR, alpha=0.3, label="Placebo range")
    ax.plot(b.index, b.values * 100, color=TREATED_COLOR, lw=2.2, marker="o", ms=4, label="Affected prices")
    ax.axhline(ev["direction"] * FULL_PASS_THROUGH * 100, color=color, ls="--", lw=1.2, label="Full pass-through")
    ax.axhline(0, color="black", lw=0.8)
    ax.axvline(-0.5, color="black", ls=":", lw=1)
    ax.set_title(f"{name}\npass-through {r['pass_through']:.0%}", fontsize=12, color=color)
    ax.set_xlabel("Months from VAT change")
axes[0].set_ylabel("Log points (≈ % vs. month before)")
axes[0].legend(loc="upper left", fontsize=8.5)
fig.suptitle("Affected prices relative to unaffected services around each VAT change", fontsize=15, fontweight="bold")
plt.tight_layout(); plt.show()

## 5. Results: How Much Was Passed On?
- **Price effect:** average change in affected prices over the 6 months from the change, relative to unaffected services
- **Pass-through:** price effect ÷ the full ~4% effect
- **Placebo p-value:** share of placebo series with an effect at least as large. Small = unlikely to be luck

In [ ]:
summary = pd.DataFrame({
    name: {"VAT change": "Cut 13.5% → 9%" if r["event"]["direction"] < 0 else "Rise 9% → 13.5%",
           "Price effect": r["betas"][r["betas"].index >= 0].mean(),
           "Pass-through": r["pass_through"],
           "Placebo p-value": r["placebo_p"],
           "Fiscal Council (2025)": BENCHMARK.get(name, np.nan),
           "Months of data after": r["post_months"]}
    for name, r in results.items()}).T

cuts = summary[summary["VAT change"].str.startswith("Cut")]["Pass-through"].astype(float)
rises = summary[summary["VAT change"].str.startswith("Rise")]["Pass-through"].astype(float)
print(f"Average pass-through of RISES: {rises.mean():.0%}   Average pass-through of CUTS: {cuts.mean():.0%}")
summary.style.format({"Price effect": lambda v: f"{v * 100:+.1f}%", "Pass-through": "{:.0%}",
                      "Placebo p-value": "{:.2f}", "Fiscal Council (2025)": "{:.0%}"}, na_rep="—")

### Chart 3: Pass-Through of Cuts vs. Rises
Each bar is one VAT change. Green = cuts, red = rises. The black diamonds are the Fiscal Council's estimates for the same changes, using more detailed data. If rises are passed on more than cuts ("rockets and feathers"), the red bars should be taller.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
names = list(summary.index)
colors = [CUT_COLOR if c.startswith("Cut") else RISE_COLOR for c in summary["VAT change"]]
bars = ax.bar(names, summary["Pass-through"].astype(float), color=colors, alpha=0.85, width=0.6)
ax.bar_label(bars, labels=[f"{v:.0%}" for v in summary["Pass-through"]], label_type="center",
             color="white", fontweight="bold", fontsize=11)     ## Inside the bars so they don't clash with the diamonds
bench = summary["Fiscal Council (2025)"].astype(float)
ax.scatter(names, bench, marker="D", color="black", s=50, zorder=3, label="Fiscal Council (2025)")
ax.axhline(1, color="black", ls="--", lw=1)
ax.text(len(names) - 0.5, 1.01, "Full pass-through", ha="right", va="bottom", fontsize=9)
ax.axhline(0, color="black", lw=0.8)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
ax.set_ylim(min(-0.2, summary["Pass-through"].astype(float).min() - 0.1), max(1.15, summary["Pass-through"].astype(float).max() + 0.15))
ax.set_title("Share of each hospitality VAT change passed on to prices")
ax.set_ylabel("Pass-through")
ax.legend(loc="upper left")
ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

### Pass-Through by Type of Service
Each affected category on its own, compared with the same controls. Categories that mix affected and unaffected items (e.g. a restaurant index that includes alcohol) will understate pass-through.

In [ ]:
by_service = pd.DataFrame({name: r["by_series"] for name, r in results.items()})
by_service.style.format("{:.0%}", na_rep="—")

## 6. Limitations
- **Public CPI sub-indices are broad.** Some mix affected and unaffected items, which pulls pass-through towards zero. The Fiscal Council study used item-level prices supplied by the CSO for this reason
- **The control group is other Irish services, not the same services abroad.** The Fiscal Council used UK prices of the same services instead. Other Irish services may respond to different cost pressures
- **2019:** the minimum wage also rose on 1 January 2019, at the same time as the VAT increase, which could add to hospitality prices
- **2020:** COVID lockdowns meant many hospitality prices weren't collected, and the standard VAT rate (which applies to several control series) was cut from 23% to 21% between September 2020 and February 2021, inside this event window. Treat this result with caution
- **2023:** high general inflation, and months of uncertainty over whether the rise would go ahead, may have spread price changes over a longer period
- **2026:** only a couple of months of data are available, so this estimate is preliminary. Hotels were not included in the cut